In [ ]:
# Install necessary packages

In [ ]:
pip install huggingface_hub
import pandas as pd 

In [ ]:
# Load and inspect Dataset

In [35]:
p = pd.read_csv("hf://datasets/TheFusion21/PokemonCards/train.csv")

print("#####COLUMNS IN DATASET#####")
print(p.columns)

print("\n")

print("#####FIRST 5 ROWS OF DATASET#####")
print(p.head())

print("\n")

print("#####INFO OF DATASET#####")
print(p.info())

print("\n")

print("#####SHAPE OF DATASET#####")
print(p.shape)

#####COLUMNS IN DATASET#####
Index(['id', 'image_url', 'caption', 'name', 'hp', 'set_name'], dtype='object')


#####FIRST 5 ROWS OF DATASET#####
        id                                       image_url  \
0    pl3-1    https://images.pokemontcg.io/pl3/1_hires.png   
1   ex12-1   https://images.pokemontcg.io/ex12/1_hires.png   
2    xy5-1    https://images.pokemontcg.io/xy5/1_hires.png   
3  mcd19-1  https://images.pokemontcg.io/mcd19/1_hires.png   
4    ex7-1    https://images.pokemontcg.io/ex7/1_hires.png   

                                             caption        name  hp  \
0  A Basic, SP Pokemon Card of type Darkness with...     Absol G  70   
1  A Stage 1 Pokemon Card of type Colorless with ...  Aerodactyl  70   
2  A Basic Pokemon Card of type Grass with the ti...      Weedle  50   
3  A Basic Pokemon Card of type Grass with the ti...    Caterpie  50   
4  A Stage 1 Pokemon Card of type Water with the ...   Azumarill  80   

                     set_name  
0             Sup

In [ ]:
# Check for Null Values

In [36]:
print("#####NULL VALUES IN DATASET PER COLUMN#####")
p.isnull().sum()

#####NULL VALUES IN DATASET PER COLUMN#####


id           0
image_url    0
caption      0
name         0
hp           0
set_name     0
dtype: int64

In [ ]:
# Check for Duplicates

In [38]:
print("#####NUMBER OF DUPLICATE ROWS IN DATASET#####")
print(p.duplicated().sum())

print("\n")

print("#####NUMBER OF DUPLICATE ROWS BASED ON 'name', 'set_name', 'caption', 'hp' COLUMNS#####")
print(p.duplicated(subset=['name', 'set_name', 'caption', 'hp']).sum())

print("\n")

print("#####SAMPLE DUPLICATE ROWS BASED ON 'name', 'set_name', 'caption' AND 'hp' COLUMNS#####")
dupes = p[p.duplicated(subset=['name', 'set_name', 'caption', 'hp'], keep=False)]
dupes.sort_values(by=['name', 'set_name', 'caption', 'hp']).head(10)

#####NUMBER OF DUPLICATE ROWS IN DATASET#####
0


#####NUMBER OF DUPLICATE ROWS BASED ON 'name', 'set_name', 'caption', 'hp' COLUMNS#####
189


#####SAMPLE DUPLICATE ROWS BASED ON 'name', 'set_name', 'caption' AND 'hp' COLUMNS#####


,id,image_url,caption,name,hp,set_name
12840,swsh11-179,https://images.pokemontcg.io/swsh11/179_hires.png,"A Basic, V Pokemon Card of type Fighting with ...",Aerodactyl V,210,Lost Origin
12841,swsh11-180,https://images.pokemontcg.io/swsh11/180_hires.png,"A Basic, V Pokemon Card of type Fighting with ...",Aerodactyl V,210,Lost Origin
466,sm6-2,https://images.pokemontcg.io/sm6/2_hires.png,A Stage 1 Pokemon Card of type Grass with the ...,Alolan Exeggutor,160,Forbidden Light
676,sm6-2a,https://images.pokemontcg.io/sm6/2a_hires.png,A Stage 1 Pokemon Card of type Grass with the ...,Alolan Exeggutor,160,Forbidden Light
1832,sm2-19,https://images.pokemontcg.io/sm2/19_hires.png,A Basic Pokemon Card of type Water with the ti...,Alolan Sandshrew,60,Guardians Rising
1912,sm2-19a,https://images.pokemontcg.io/sm2/19a_hires.png,A Basic Pokemon Card of type Water with the ti...,Alolan Sandshrew,60,Guardians Rising
2053,sm2-21,https://images.pokemontcg.io/sm2/21_hires.png,A Basic Pokemon Card of type Water with the ti...,Alolan Vulpix,60,Guardians Rising
2208,sm2-21a,https://images.pokemontcg.io/sm2/21a_hires.png,A Basic Pokemon Card of type Water with the ti...,Alolan Vulpix,60,Guardians Rising
4077,sm75-40,https://images.pokemontcg.io/sm75/40_hires.png,A Stage 1 Pokemon Card of type Dragon with the...,Altaria,80,Dragon Majesty
4204,sm75-40a,https://images.pokemontcg.io/sm75/40a_hires.png,A Stage 1 Pokemon Card of type Dragon with the...,Altaria,80,Dragon Majesty


In [ ]:
# Create New DataFrame

In [ ]:
llava_df = pd.DataFrame()

llava_df['image'] = p["image_url"]
llava_df['input'] = "Can you write a detailed description of the following Pokemon card?"
llava_df['output'] = p["caption"]
llava_df["type"] = "conv"

llava_df_without_dupes = llava_df.drop_duplicates(subset=['output'])

print("#####SHAPE OF NEW DATAFRAME WITHOUT DUPLICATES#####")
print(llava_df_without_dupes.shape)

print("\n")

print("#####NEW DATAFRAME COLUMNS#####")
print(llava_df.columns)

print("\n")

print("#####FIRST ROW OF NEW DATAFRAME#####")
print(llava_df.iloc[0])

#####SHAPE OF NEW DATAFRAME WITHOUT DUPLICATES#####
(12950, 4)


#####NEW DATAFRAME COLUMNS#####
Index(['image', 'input', 'output', 'type'], dtype='object')


#####FIRST ROW OF NEW DATAFRAME#####
image          https://images.pokemontcg.io/pl3/1_hires.png
input     Can you write a detailed description of the fo...
output    A Basic, SP Pokemon Card of type Darkness with...
type                                                   conv
Name: 0, dtype: object


In [45]:
# Adjust URLs for direct image access
def adjust_image_url(url):
    return "/".join(url.split("/")[-2:]).split("_")[0]

llava_df['image'] = llava_df['image'].apply(adjust_image_url)   
llava_df_without_dupes['image'] = llava_df_without_dupes['image'].apply(adjust_image_url)
print("#####ADJUSTED IMAGE URL SAMPLE#####")
print(llava_df['image'].head())

#####ADJUSTED IMAGE URL SAMPLE#####
0      pl3/1
1     ex12/1
2      xy5/1
3    mcd19/1
4      ex7/1
Name: image, dtype: object


/var/folders/8k/s6fjz5gn2qgfs589f5g1l6940000gn/T/ipykernel_41040/1689942795.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  llava_df_without_dupes['image'] = llava_df_without_dupes['image'].apply(adjust_image_url)


In [47]:
# split into train and test sets
llava_df_TRAIN = llava_df[:10000]
llava_df_TEST = llava_df[10000:]

llava_df_without_dupes_TRAIN = llava_df_without_dupes[:10000]
llava_df_without_dupes_TEST = llava_df_without_dupes[10000:]

print("#####SHAPE OF TRAIN AND TEST SETS#####")
print("TRAIN SET SHAPE WITH DUPES: ", llava_df_TRAIN.shape)
print("TEST SET SHAPE WITH DUPES: ", llava_df_TEST.shape)
print("TRAIN SET SHAPE WITHOUT DUPES: ", llava_df_without_dupes_TRAIN.shape)
print("TEST SET SHAPE WITHOUT DUPES: ", llava_df_without_dupes_TEST.shape)

#####SHAPE OF TRAIN AND TEST SETS#####
TRAIN SET SHAPE WITH DUPES:  (10000, 4)
TEST SET SHAPE WITH DUPES:  (3139, 4)
TRAIN SET SHAPE WITHOUT DUPES:  (10000, 4)
TEST SET SHAPE WITHOUT DUPES:  (2950, 4)


In [48]:
# Save New DataFrame to jsonl
llava_df_TRAIN.to_json("pokemon_llava_dataset_train.jsonl", orient="records", lines=True)
llava_df_without_dupes_TRAIN.to_json("pokemon_llava_dataset_without_dupes_train.jsonl", orient="records", lines=True)

llava_df_TEST.to_json("pokemon_llava_dataset_test.jsonl", orient="records", lines=True)    
llava_df_without_dupes_TEST.to_json("pokemon_llava_dataset_without_dupes_test.jsonl", orient="records", lines=True)
